## Foundry Agent with Code Interpreter Capability

![lab_flow](.\Assets\FoundryAgent_CodeInterpreter.png)

### Installing Required Libraries

In [62]:

%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity pandas matplotlib openpyxl

Note: you may need to restart the kernel to use updated packages.


### Setting Up the Environment Variables

In [63]:
import os
from dotenv import load_dotenv
from azure.identity import ClientSecretCredential
from azure.ai.projects import AIProjectClient

# Load variables safely from your local environment file
load_dotenv()

# 2. Instantiate the credential using the verified memory strings
credential = ClientSecretCredential(
    tenant_id=os.environ["AZURE_TENANT_ID"],
    client_id=os.environ["AZURE_CLIENT_ID"],
    client_secret=os.environ["AZURE_CLIENT_SECRET"]
)

# 3. Spin up your project client mapping
project_client = AIProjectClient(
    endpoint=os.getenv("FOUNDRY_PROJECT_ENDPOINT"),
    credential=credential
)

openai_client = project_client.get_openai_client()

print("🎉 SPN Context explicitly injected and authenticated successfully!")

🎉 SPN Context explicitly injected and authenticated successfully!


### Setting Up the Foundry Project Client

In [64]:
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=AzureKeyCredential(foundry_project_key)
)

### Creating the OpenAI Client

In [65]:
from openai import OpenAI

base_endpoint = foundry_project_endpoint.split("/v1")[0] + "/v1"

openai_client = OpenAI(
    base_url=base_endpoint,
    api_key=foundry_project_key
)
print("🚀 Client mapped successfully via standard proxy routing protocol.")

🚀 Client mapped successfully via standard proxy routing protocol.


### CSV File Upload

In [66]:
# Upload the CSV file using the authenticated OpenAI data plane client
file = openai_client.files.create(
    purpose="assistants",
    file=open("./electronics_products.csv", "rb")
)

print(f"✅ File uploaded successfully! (id: {file.id})")

BadRequestError: Error code: 400 - {'error': {'code': 'BadRequest', 'message': 'Missing required query parameter: api-version'}}

### Creating Agent with the Code Interpreter Tool

In [ ]:
from azure.ai.projects.models import CodeInterpreterTool, CodeInterpreterToolAuto, PromptAgentDefinition

agent_name = "code-interpreter-agent"

# Provision the agent using your active, SPN-authenticated project client
agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful AI Assistant with code interpreter capabilities.",
        tools=[
            CodeInterpreterTool(
                container=CodeInterpreterToolAuto(
                    file_ids=[file.id]
                )
            )
        ]
    )
)

# Printing the verified agent configuration parameters
print(f"🎉 Agent created successfully! (id: {agent.id}, name: {agent.name}, version: {agent.version})")

🎉 Agent created successfully! (id: code-interpreter-agent:1, name: code-interpreter-agent, version: 1)


### Creating a Conversation Object for the Agent Chat System

In [ ]:
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

Created conversation with id: conv_4d9c498f5e73abb600e9CjCi5nyphoip5YDP6pmOkKLliDqomu


### Calling Our Agent

In [ ]:
import time

print("⏳ Cooling down the inference rate window for 10 seconds...")
time.sleep(10)

# Re-run your original call
response = openai_client.responses.create(
    conversation=conversation.id,
    input="Could you please create a column chart with products on the x-axis and their respective prices on the y-axis?",
    extra_body={
        "agent": {
            "name": agent.name,
            "type": "agent_reference"
        }
    }
)
print(f"Response completed with id: {response.id}")

⏳ Cooling down the inference rate window for 10 seconds...
Response completed with id: resp_4d9c498f5e73abb6006a3fb9b2e5ec8190bf0b27d1c8aaf085


### Extracting file information from response annotations

In [ ]:
file_id = ""
filename = ""
container_id = ""

# Get the last message which should contain file citations
last_message = response.output[-1]  # ResponseOutputMessage
if last_message.type == "message":
        # Get the last content item (contains the file annotations)
        text_content = last_message.content[-1]  # ResponseOutputText
        if text_content.type == "output_text":
            # Get the last annotation (most recent file)
            if text_content.annotations:
                file_citation = text_content.annotations[-1]  # AnnotationContainerFileCitation
                if file_citation.type == "container_file_citation":
                    file_id = file_citation.file_id
                    filename = file_citation.filename
                    container_id = file_citation.container_id
                    print(f"Found generated file: {filename} (ID: {file_id})")

# Download the generated file if available
if file_id and filename:
        file_content = openai_client.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
        with open(filename, "wb") as f:
            f.write(file_content.read())
            print(f"File {filename} downloaded successfully.")
        print(f"File ready for download: {filename}")
else:
        print("No file generated in response")

Found generated file: product_price_column_chart.png (ID: cfile_6a3fb9b962a4819095131da247bda619)
File product_price_column_chart.png downloaded successfully.
File ready for download: product_price_column_chart.png
